## Model predictions
Import libraries

In [ ]:
import os
import json
import utils
import numpy as np
import tensorflow as tf

Define inputs

In [ ]:
model_name = "meno_nocorr"

In [ ]:
# Models
models = {
  "meno": "./../03.merged/run01/saving/plasma_cr/SavedModel/",
  "meno_nocorr": "./../02.states/run01.nocorr/saving/plasma_cr_states/SavedModel/"
}
# Paths
path_to_datagen = "./../00.datagen.r03/"
path_to_data = path_to_datagen + "/data.v8/"
paths = {
  "data": path_to_data + "/test/",
  "model": models[model_name],
  "indices": path_to_data + "/indices.json",
  "inp_space": path_to_datagen + "/datagen/runs/samples/inp_space.hdf5",
  "saving": f"./output/visual/{model_name}"
}
os.makedirs(paths["saving"], exist_ok=True)
# Cases
cases = [29, 31, 32]

In [ ]:
# Time grid
tlim = [1e-8, 1e-6]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Define data indices

In [ ]:
# Data indices
with open(path_to_data + "/indices.json") as file:
  _indices = json.load(file)
if (model_name == "meno"):
  indices = {
    "x": [
      _indices["rho_0"],
      _indices["Th_0"],
      _indices["Te_0"],
      _indices["t"]
    ],
    "y": [
      _indices["w"],
      _indices["Th"],
      _indices["Te"]
    ]
  }
elif (model_name == "meno_nocorr"):
  indices = {
    "x": [
      _indices["rho_0"],
      _indices["Th_0"],
      _indices["Te_0"],
      _indices["t"],
      [_indices[k][0] for k in ("n_em", "n_Ar1", "Th", "Te")]
    ],
    "y": [
      _indices["w"]
    ]
  }
else:
  raise ValueError()

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])

Plot true vs. predicted solutions

In [ ]:
for (icase, idata) in data.items():
  idata = idata.to_numpy()
  case_nb = int(icase.split("_")[-1])
  if ((cases is not None) and (case_nb not in cases)):
    continue
  _, y_true, y_pred = utils.predict(model, idata, indices, t, t_index=3)
  utils.plot_test(
    path=paths["saving"],
    x=t,
    y=y_true[:,:-2],
    y_pred=y_pred[:,:-2],
    figname=icase+"_species_log",
    figext=["pdf", "pgf"],
    labels=["$t$ [s]", "$w$"],
    scales=["log", "log"],
    xlim=tlim,
    save=True,
    show=False
  )